## Example 1: Basic Import and Inspection\n
\n
Basic import test and inspection of greenlet package structure.\n

In [ ]:
# Example 1: Basic Greenlet Creation and Switching
from greenlet import greenlet

print("Example 1: Basic Greenlet Creation and Switching")
print("=" * 50)

# Simple greenlet example
def test1():
    print("  In test1: Starting")
    gr2.switch()
    print("  In test1: Resumed after gr2")
    return "test1 complete"

def test2():
    print("  In test2: Starting")
    gr1.switch()
    print("  In test2: Resumed after gr1")
    return "test2 complete"

# Create greenlets
gr1 = greenlet(test1)
gr2 = greenlet(test2)

print("\n1. Initial state:")
print(f"   gr1 is dead: {gr1.dead}")
print(f"   gr2 is dead: {gr2.dead}")

print("\n2. Switching to gr1:")
result = gr1.switch()

print("\n3. After first switch:")
print(f"   gr1 is dead: {gr1.dead}")
print(f"   gr2 is dead: {gr2.dead}")

print("\n4. Switching to gr2 to complete:")
gr2.switch()

print("\n5. Final state:")
print(f"   gr1 is dead: {gr1.dead}")
print(f"   gr2 is dead: {gr2.dead}")

# Test parent-child relationship
print("\n6. Testing parent greenlet:")
main = greenlet.getcurrent()
print(f"   Current greenlet: {main}")
print(f"   gr1 parent: {gr1.parent}")

print("\n✅ Example 1 completed successfully!")

## Example 2: Core Functionality Test\n
\n
Test core functionality of greenlet package.\n

In [ ]:
# Example 2: Passing Values Between Greenlets
from greenlet import greenlet

print("Example 2: Passing Values Between Greenlets")
print("=" * 45)

# Greenlet that processes data
def producer():
    print("\n  Producer: Starting production")
    for i in range(5):
        value = i * 10
        print(f"  Producer: Producing {value}")
        received = gr_main.switch(value)
        print(f"  Producer: Received acknowledgment: {received}")
    print("  Producer: Done")
    return "Producer finished"

# Create producer greenlet
gr_producer = greenlet(producer)
gr_main = greenlet.getcurrent()

print("1. Starting producer:")
values_received = []

for _ in range(5):
    value = gr_producer.switch()
    values_received.append(value)
    print(f"  Main: Received {value}, sending ACK")
    gr_producer.switch(f"ACK-{value}")

print(f"\n2. All values received: {values_received}")

# Example with bidirectional communication
print("\n\n3. Bidirectional Communication:")
print("-" * 40)

def worker(name):
    print(f"  {name}: Started")
    received = None
    while True:
        print(f"  {name}: Waiting for data...")
        received = gr_main.switch(f"{name} ready")
        if received == "STOP":
            print(f"  {name}: Stopping")
            break
        print(f"  {name}: Processing {received}")
    return f"{name} completed"

gr_worker = greenlet(worker)
status = gr_worker.switch("Worker1")
print(f"  Main: Got status: {status}")

# Send work items
for item in ["Task-A", "Task-B", "Task-C"]:
    print(f"  Main: Sending {item}")
    status = gr_worker.switch(item)
    print(f"  Main: Got status: {status}")

# Stop worker
print("  Main: Sending STOP signal")
result = gr_worker.switch("STOP")
print(f"  Main: Worker result: {result}")

print("\n✅ Example 2 completed successfully!")

## Example 3: Error Handling and Edge Cases\n
\n
Test error handling and edge cases for greenlet.\n

In [ ]:
# Example 3: Exception Handling in Greenlets
from greenlet import greenlet, GreenletExit

print("Example 3: Exception Handling in Greenlets")
print("=" * 45)

# Test normal exception propagation
print("1. Normal exception handling:")
print("-" * 40)

def risky_operation():
    print("  Greenlet: Starting risky operation")
    gr_main.switch("checkpoint 1")
    print("  Greenlet: About to raise exception")
    raise ValueError("Something went wrong!")

gr_risky = greenlet(risky_operation)
gr_main = greenlet.getcurrent()

try:
    result = gr_risky.switch()
    print(f"  Main: Got checkpoint: {result}")
    result = gr_risky.switch()
except ValueError as e:
    print(f"  Main: Caught exception: {e}")

print(f"  Greenlet is dead: {gr_risky.dead}")

# Test GreenletExit exception
print("\n\n2. GreenletExit handling:")
print("-" * 40)

def cleanable_task():
    try:
        print("  Task: Starting")
        gr_main.switch("started")
        print("  Task: Working...")
        gr_main.switch("working")
        print("  Task: This won't execute")
    except GreenletExit:
        print("  Task: Cleaning up on GreenletExit")
        raise

gr_task = greenlet(cleanable_task)
print(f"  Main: First switch")
gr_task.switch()

print(f"  Main: Second switch")
gr_task.switch()

print(f"  Main: Throwing GreenletExit")
try:
    gr_task.throw(GreenletExit)
except GreenletExit:
    print(f"  Main: GreenletExit propagated to parent")

print(f"  Task is dead: {gr_task.dead}")

# Test exception in parent
print("\n\n3. Multiple greenlets with exceptions:")
print("-" * 40)

results = []

def safe_worker(worker_id):
    try:
        print(f"  Worker {worker_id}: Starting")
        gr_main.switch(f"W{worker_id} started")
        
        if worker_id == 2:
            raise RuntimeError(f"Worker {worker_id} failed")
        
        print(f"  Worker {worker_id}: Completed")
        return f"W{worker_id} success"
    except Exception as e:
        print(f"  Worker {worker_id}: Caught {type(e).__name__}: {e}")
        return f"W{worker_id} error"

workers = [greenlet(lambda wid=i: safe_worker(wid)) for i in range(1, 4)]

for i, worker in enumerate(workers, 1):
    try:
        status = worker.switch()
        print(f"  Main: Status from worker {i}: {status}")
        result = worker.switch()
        results.append(result)
        print(f"  Main: Result from worker {i}: {result}")
    except Exception as e:
        results.append(f"W{i} exception: {e}")
        print(f"  Main: Caught exception from worker {i}: {e}")
        raise

print(f"\n  All results: {results}")

print("\n✅ Example 3 completed successfully!")

## Example 4: Integration and Usage Patterns\n
\n
Test integration patterns and common usage of greenlet.\n

In [ ]:
# Example 4: Parent-Child Greenlet Relationships
from greenlet import greenlet

print("Example 4: Parent-Child Greenlet Relationships")
print("=" * 48)

# Demonstrate parent-child hierarchy
print("1. Greenlet hierarchy:")
print("-" * 40)

main = greenlet.getcurrent()
print(f"  Main greenlet: {main}")
print(f"  Main parent: {main.parent}")
print(f"  Main is dead: {main.dead}")

def child_function():
    current = greenlet.getcurrent()
    print(f"  Child greenlet: {current}")
    print(f"  Child parent: {current.parent}")
    print(f"  Child parent is main: {current.parent is main}")
    return "child done"

child = greenlet(child_function)
print(f"\n  Created child: {child}")
print(f"  Child parent: {child.parent}")

result = child.switch()
print(f"  Child returned: {result}")

# Create greenlet with explicit parent
print("\n\n2. Explicit parent specification:")
print("-" * 40)

def grandchild_function():
    current = greenlet.getcurrent()
    print(f"  Grandchild: {current}")
    print(f"  Grandchild parent: {current.parent}")
    return "grandchild done"

def parent_function():
    current = greenlet.getcurrent()
    print(f"  Parent: {current}")
    print(f"  Parent parent: {current.parent}")
    
    # Create grandchild with parent as explicit parent
    grandchild = greenlet(grandchild_function, parent=current)
    print(f"  Created grandchild with parent: {grandchild.parent}")
    
    result = grandchild.switch()
    print(f"  Grandchild returned: {result}")
    return "parent done"

parent = greenlet(parent_function)
result = parent.switch()
print(f"  Parent returned: {result}")

# Demonstrate switching to specific parent
print("\n\n3. Switching back to parent:")
print("-" * 40)

def level3():
    print("  Level 3: Executing")
    print(f"  Level 3 parent: {greenlet.getcurrent().parent}")
    # Switch to parent (level2)
    greenlet.getcurrent().parent.switch("from level 3")
    print("  Level 3: Resumed")
    return "level 3 complete"

def level2():
    print("  Level 2: Executing")
    gr3 = greenlet(level3)
    result = gr3.switch()
    print(f"  Level 2: Received '{result}' from level 3")
    return "level 2 complete"

def level1():
    print("  Level 1: Executing")
    gr2 = greenlet(level2)
    result = gr2.switch()
    print(f"  Level 1: Received '{result}' from level 2")
    return "level 1 complete"

gr1 = greenlet(level1)
final = gr1.switch()
print(f"  Main: Received '{final}'")

# Test getcurrent()
print("\n\n4. Using getcurrent():")
print("-" * 40)

def show_current():
    current = greenlet.getcurrent()
    main_ref = greenlet.getcurrent().parent
    print(f"  Inside greenlet: {current}")
    print(f"  Main greenlet: {main_ref}")
    print(f"  Are they different: {current is not main_ref}")

gr = greenlet(show_current)
gr.switch()

print(f"  In main, current is: {greenlet.getcurrent()}")
print(f"  Main is main: {greenlet.getcurrent() is main}")

print("\n✅ Example 4 completed successfully!")

## Example 5: Comprehensive Feature Test\n
\n
Comprehensive test of greenlet features and capabilities.\n

In [ ]:
# Example 5: Practical Use Cases - Cooperative Multitasking
from greenlet import greenlet

print("Example 5: Practical Use Cases - Cooperative Multitasking")
print("=" * 58)

# Simulate a simple task scheduler
print("1. Simple Round-Robin Task Scheduler:")
print("-" * 45)

class Task:
    def __init__(self, name, work_units):
        self.name = name
        self.work_units = work_units
        self.completed = 0
        self.greenlet = greenlet(self.run)
    
    def run(self):
        while self.completed < self.work_units:
            self.completed += 1
            print(f"  {self.name}: Completed unit {self.completed}/{self.work_units}")
            # Yield control back to scheduler
            scheduler.switch()
        return f"{self.name} finished"

# Create tasks
tasks = [
    Task("TaskA", 3),
    Task("TaskB", 4),
    Task("TaskC", 2)
]

scheduler = greenlet.getcurrent()
active_tasks = tasks.copy()

print("  Starting tasks...")
while active_tasks:
    completed = []
    for task in active_tasks:
        if task.greenlet.dead:
            completed.append(task)
        else:
            result = task.greenlet.switch()
            if task.greenlet.dead:
                print(f"  Scheduler: {result}")
                completed.append(task)
    
    # Remove completed tasks
    for task in completed:
        active_tasks.remove(task)

print("  All tasks completed!")

# Producer-Consumer pattern
print("\n\n2. Producer-Consumer Pattern:")
print("-" * 35)

buffer = []
buffer_size = 3

def producer():
    for i in range(8):
        while len(buffer) >= buffer_size:
            print(f"  Producer: Buffer full, waiting...")
            consumer_gr.switch()
        
        item = f"Item-{i}"
        buffer.append(item)
        print(f"  Producer: Produced {item} (buffer: {len(buffer)})")
        consumer_gr.switch()
    
    print("  Producer: Done producing")

def consumer():
    consumed = []
    while True:
        if not buffer:
            # Check if producer is done
            if producer_gr.dead:
                break
            print(f"  Consumer: Buffer empty, waiting...")
            producer_gr.switch()
            continue
        
        item = buffer.pop(0)
        consumed.append(item)
        print(f"  Consumer: Consumed {item} (buffer: {len(buffer)})")
        
        if not producer_gr.dead:
            producer_gr.switch()
    
    print(f"  Consumer: Done consuming, total: {len(consumed)}")
    return consumed

producer_gr = greenlet(producer)
consumer_gr = greenlet(consumer)

result = producer_gr.switch()
consumed_items = consumer_gr.switch()

print(f"  Final buffer state: {buffer}")
print(f"  Items consumed: {len(consumed_items)}")

# Cooperative generator pattern
print("\n\n3. Cooperative Generators:")
print("-" * 30)

def number_generator(start, end):
    """Generate numbers and yield control"""
    current = greenlet.getcurrent()
    main = current.parent
    
    for i in range(start, end):
        print(f"  Generator: Yielding {i}")
        main.switch(i)
    
    return "generator done"

# Create generators
gen1 = greenlet(lambda: number_generator(1, 4))
gen2 = greenlet(lambda: number_generator(10, 13))

results = []

# Interleave execution of two generators
print("  Interleaving two generators:")
for _ in range(3):
    val1 = gen1.switch()
    results.append(val1)
    print(f"  Main: Got {val1} from gen1")
    
    val2 = gen2.switch()
    results.append(val2)
    print(f"  Main: Got {val2} from gen2")

print(f"  Collected values: {results}")
print(f"  Gen1 is dead: {gen1.dead}")
print(f"  Gen2 is dead: {gen2.dead}")

# State machine with greenlets
print("\n\n4. State Machine Implementation:")
print("-" * 38)

class StateMachine:
    def __init__(self):
        self.current_state = None
        self.main = greenlet.getcurrent()
    
    def state_idle(self):
        print("  State: IDLE")
        command = self.main.switch("ready")
        
        if command == "start":
            self.current_state.switch()  # Return to switch to next state
        return "stopped"
    
    def state_running(self):
        print("  State: RUNNING")
        for i in range(3):
            print(f"    Working... {i+1}/3")
            self.main.switch(f"progress-{i+1}")
        
        self.current_state.switch()  # Move to next state
    
    def state_complete(self):
        print("  State: COMPLETE")
        self.main.switch("done")
        return "finished"
    
    def run(self):
        # Create state greenlets
        idle_gr = greenlet(self.state_idle)
        running_gr = greenlet(self.state_running)
        complete_gr = greenlet(self.state_complete)
        
        # Start in idle state
        self.current_state = idle_gr
        status = idle_gr.switch()
        print(f"  Main: Status = {status}")
        
        # Send start command
        status = idle_gr.switch("start")
        
        # Transition to running
        self.current_state = running_gr
        status = running_gr.switch()
        print(f"  Main: Status = {status}")
        
        status = running_gr.switch()
        print(f"  Main: Status = {status}")
        
        status = running_gr.switch()
        print(f"  Main: Status = {status}")
        
        # Transition to complete
        self.current_state = complete_gr
        status = complete_gr.switch()
        print(f"  Main: Status = {status}")

sm = StateMachine()
sm.run()

print("\n✅ Example 5 completed successfully!")